# Master's Thesis Evaluation & Experimental Results
## Human Oversight and Human-in-the-Loop Control Engine for Autonomous AI Agents

This interactive notebook provides the statistical analysis, evaluation metrics, and publication-ready figures for the experimental chapter of the Master's thesis.

### Objectives Evaluated:
1. **Task Effectiveness & Gap Analysis**: Technical completion rate vs. Semantic correctness.
2. **Oversight Quality**: Escalation confusion matrix, Precision, Recall, F1-Score, and Human Intervention Rate.
3. **Pillar Breakdown**: Distribution of triggers across Evidence, Trajectory, Semantic Judge, and Static Policy pillars.
4. **System Overhead & Latency**: Actor turns, tool calls, governance passes, and human review duration.
5. **Experiment Comparison**: Comparative assessment across ingested experiment IDs, when more than one is present.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Ensure project modules are importable
project_root = Path("..").resolve() if Path("..").resolve().name == "tesi" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from notebooks.thesis_plotting import (
    setup_thesis_style,
    load_evaluation_summary,
    load_run_level_frames,
    plot_technical_vs_semantic_gap,
    plot_escalation_confusion_matrix,
    plot_governance_pillars_and_reasons,
    plot_overhead_and_efficiency,
    plot_experiment_comparison,
    export_all_thesis_figures,
)

setup_thesis_style()
print("Thesis Evaluation environment initialized successfully.")

## 1. Ingestion & Data Loading
Load the evaluated runs directly from the SQLite database `artifacts/evaluation/layer5_evaluation.sqlite3`,
via the canonical `build_evaluation_summary` used by the Layer 5 SVG report - this notebook never
recomputes a rate on its own. Ingest a real campaign first (see `notebooks/README.md`); there is no
synthetic fallback here anymore.

In [ ]:
db_path = project_root / "artifacts" / "evaluation" / "layer5_evaluation.sqlite3"
summary = load_evaluation_summary(db_path)
run_frames = load_run_level_frames(db_path)

if summary.total_runs == 0:
    raise ValueError(
        f"No runs found in {db_path}. Ingest a real evaluation campaign first "
        "(see notebooks/README.md) before running this notebook."
    )

print(f"Total Runs: {summary.total_runs}")
print(f"Governance Passes: {summary.total_governance_events}")
print(f"Judged Runs: {summary.judged_runs}")

run_frames["runs"].head(5)

## 2. Technical Completion vs. Semantic Correctness (Gap Analysis)
A core claim in the thesis is that an agent reaching technical termination (`completed`) does not necessarily satisfy the goal. The chart below highlights this gap.

In [ ]:
fig1 = plot_technical_vs_semantic_gap(summary)
plt.show()

## 3. Human Oversight Confusion Matrix & Escalation Performance
Evaluating the decision engine (OPA + Pilastri + Semantic Judge):
- **True Positive**: Review was required and legitimately needed (risk prevented).
- **False Positive**: Review was required but was unnecessary (over-cautious policy).
- **False Negative**: Review was omitted on an unsafe or erroneous candidate (safety failure).
- **True Negative**: Autonomy was safely granted.

In [ ]:
fig2 = plot_escalation_confusion_matrix(summary)
plt.show()

## 4. Governance Pillars & Reason Code Breakdown
Analyzing which pillar of the oversight architecture prompted escalations and policy directives.

In [ ]:
fig3 = plot_governance_pillars_and_reasons(summary)
plt.show()

## 5. Computational Overhead & Review Latency
Distribution of actor turns, tool interactions, governance cycles, and operator response times.

In [ ]:
fig4 = plot_overhead_and_efficiency(run_frames)
plt.show()

## 6. Comparison Across Experiments
Compares technical completion, task accuracy and human intervention rate across whichever
`experiment_id`s are actually present in the database. Shown only when at least two experiments
have been ingested - there is no fabricated ablation design here.

In [ ]:
fig5 = plot_experiment_comparison(db_path)
if fig5 is not None:
    plt.show()
else:
    print("Only one experiment_id in the database - nothing to compare yet.")

## 7. Export High-Resolution Figures for LaTeX
Saves all figures in vector PDF (`.pdf`) and 300 DPI PNG (`.png`) for direct inclusion in the LaTeX thesis document.

In [ ]:
figures_out_dir = project_root / "figures" / "thesis"
exported_files = export_all_thesis_figures(db_path, figures_out_dir)

print(f"Successfully generated {len(exported_files)} figure files in: {figures_out_dir}")
for f in exported_files:
    print(f"  - {f.name}")